In [0]:
import requests
import pandas as pd
import json
import time
import uuid

from pathlib import Path
from datetime import datetime, timezone


# ============================================================
# PATH CONFIGURATION
# ============================================================

BASE_DIR = Path("/Volumes/fooddata/rawdata/data")

SOURCE_DIR = BASE_DIR / "source"
LANDING_DIR = BASE_DIR / "landing"

BIGBASKET_FILE = SOURCE_DIR / "BigBasketProducts.csv"

BIGBASKET_LANDING_DIR = LANDING_DIR / "bigbasket"
OFF_LANDING_DIR = LANDING_DIR / "open_food_facts"
OPEN_PRICES_LANDING_DIR = LANDING_DIR / "open_prices"

# Create required directories
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
LANDING_DIR.mkdir(parents=True, exist_ok=True)
BIGBASKET_LANDING_DIR.mkdir(parents=True, exist_ok=True)
OFF_LANDING_DIR.mkdir(parents=True, exist_ok=True)
OPEN_PRICES_LANDING_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIGURATION
# ============================================================

OFF_PAGE_SIZE = 20
OFF_MAX_PAGES = 25

OPEN_PRICES_LIMIT = 100

HTTP_TIMEOUT = 60
MAX_RETRIES = 4

USER_AGENT = "MessyMartAI/1.0 (data engineering portfolio project)"

RUN_ID = str(uuid.uuid4())

MANIFEST_FILE = LANDING_DIR / "ingestion_manifest.csv"


# ============================================================
# HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "application/json"
})


# ============================================================
# HTTP REQUEST WITH RETRIES
# ============================================================

def get_with_retry(url, params=None):

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = session.get(
                url,
                params=params,
                timeout=HTTP_TIMEOUT
            )

            if response.status_code == 200:
                return response

            if response.status_code in [429, 500, 502, 503, 504]:

                if attempt < MAX_RETRIES:

                    wait_time = 10 * attempt

                    print(
                        f"Request failed with status "
                        f"{response.status_code}. "
                        f"Retrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                    continue

            print(
                f"Request failed with status "
                f"{response.status_code}"
            )

            return None

        except requests.RequestException as error:

            print(
                f"Request error on attempt "
                f"{attempt}/{MAX_RETRIES}: {error}"
            )

            if attempt < MAX_RETRIES:

                wait_time = 10 * attempt

                print(
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                return None

    return None


# ============================================================
# OPEN FOOD FACTS
# ============================================================

def fetch_open_food_facts(
    page_size=OFF_PAGE_SIZE,
    max_pages=OFF_MAX_PAGES
):

    url = "https://world.openfoodfacts.org/api/v2/search"

    fields = [
        "code",
        "product_name",
        "brands",
        "categories",
        "categories_tags",
        "quantity",
        "packaging",
        "ingredients_text",
        "nutriscore_grade",
        "nutriments",
        "countries",
        "countries_tags",
        "image_url",
        "last_modified_t"
    ]

    products = []

    print("\nFetching Open Food Facts data...")

    for page in range(1, max_pages + 1):

        params = {
            "countries_tags_en": "india",
            "page": page,
            "page_size": page_size,
            "fields": ",".join(fields)
        }

        response = get_with_retry(
            url,
            params=params
        )

        if response is None:
            print(f"Failed to fetch page {page}.")
            break

        data = response.json()

        page_products = data.get("products", [])

        if not page_products:
            print(
                f"No products returned on page {page}. "
                f"Stopping."
            )
            break

        for product in page_products:

            product["source"] = "open_food_facts"

            product["ingested_at"] = (
                datetime.now(timezone.utc).isoformat()
            )

            products.append(product)

        print(
            f"Page {page}/{max_pages}: "
            f"{len(page_products)} products"
        )

        if page < max_pages:
            time.sleep(15)

    # Save raw API response
    output_file = (
        OFF_LANDING_DIR /
        "open_food_facts_india.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            products,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"\nOpen Food Facts records: "
        f"{len(products)}"
    )

    print(
        f"Saved to: {output_file}"
    )

    return products


# ============================================================
# OPEN PRICES
# ============================================================

def fetch_open_prices(
    limit=OPEN_PRICES_LIMIT
):

    url = "https://prices.openfoodfacts.org/api/v1/prices"

    params = {
        "page_size": limit
    }

    print("\nFetching Open Prices data...")

    response = get_with_retry(
        url,
        params=params
    )

    if response is None:
        print("Failed to fetch Open Prices data.")
        return []

    data = response.json()

    if isinstance(data, dict):

        prices = data.get(
            "items",
            data.get("prices", [])
        )

    elif isinstance(data, list):

        prices = data

    else:

        prices = []

    for price in prices:

        price["source"] = "open_prices"

        price["ingested_at"] = (
            datetime.now(timezone.utc).isoformat()
        )

    output_file = (
        OPEN_PRICES_LANDING_DIR /
        "open_prices.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            prices,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Open Prices records: "
        f"{len(prices)}"
    )

    print(
        f"Saved to: {output_file}"
    )

    return prices


# ============================================================
# BIGBASKET
# ============================================================

def load_bigbasket(csv_path):

    csv_path = Path(csv_path)

    if not csv_path.exists():

        print(
            "\nBigBasket CSV not found."
        )

        print(
            "Expected file:"
        )

        print(csv_path)

        return None

    print(
        f"\nLoading BigBasket CSV:"
    )

    print(csv_path)

    df = pd.read_csv(csv_path)

    # Add ingestion metadata
    df["source"] = "bigbasket"

    df["ingested_at"] = (
        datetime.now(timezone.utc).isoformat()
    )

    # Save ingested copy to landing zone
    output_file = (
        BIGBASKET_LANDING_DIR /
        "bigbasket_raw.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"BigBasket rows: {len(df)}"
    )

    print(
        f"BigBasket columns: {len(df.columns)}"
    )

    print(
        f"Saved to: {output_file}"
    )

    print(
        "\nColumns:"
    )

    print(
        list(df.columns)
    )

    return df


# ============================================================
# INSPECT OPEN FOOD FACTS
# ============================================================

def inspect_open_food_facts(products):

    if not products:

        print(
            "\nNo Open Food Facts data to inspect."
        )

        return

    df = pd.json_normalize(products)

    print(
        "\nOpen Food Facts inspection:"
    )

    print(
        f"Rows: {len(df)}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )

    print(
        "\nFields:"
    )

    print(
        list(df.columns)
    )

    print(
        "\nSample:"
    )

    print(
        df.head()
    )


# ============================================================
# INSPECT OPEN PRICES
# ============================================================

def inspect_open_prices(prices):

    if not prices:

        print(
            "\nNo Open Prices data to inspect."
        )

        return

    df = pd.json_normalize(prices)

    print(
        "\nOpen Prices inspection:"
    )

    print(
        f"Rows: {len(df)}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )

    print(
        "\nFields:"
    )

    print(
        list(df.columns)
    )

    print(
        "\nSample:"
    )

    print(
        df.head()
    )


# ============================================================
# BASIC INGESTION VALIDATION
# ============================================================

def validate_ingestion(
    source,
    record_count
):

    if record_count > 0:

        print(
            f"{source}: SUCCESS "
            f"({record_count} records)"
        )

        return "SUCCESS"

    print(
        f"{source}: FAILED "
        f"(0 records)"
    )

    return "FAILED"


# ============================================================
# INGESTION MANIFEST
# ============================================================

def create_ingestion_manifest(results):

    manifest_df = pd.DataFrame(results)

    manifest_df.to_csv(
        MANIFEST_FILE,
        index=False
    )

    print(
        "\nIngestion manifest:"
    )

    print(
        manifest_df
    )

    print(
        f"\nManifest saved to: "
        f"{MANIFEST_FILE}"
    )


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():

    pipeline_start = datetime.now(
        timezone.utc
    )

    print(
        "=" * 60
    )

    print(
        "MessyMart AI - Data Ingestion Pipeline"
    )

    print(
        "=" * 60
    )

    print(
        f"Run ID: {RUN_ID}"
    )

    print(
        f"Pipeline started: "
        f"{pipeline_start.isoformat()}"
    )

    results = []

    # --------------------------------------------------------
    # OPEN FOOD FACTS
    # --------------------------------------------------------

    source_start = datetime.now(
        timezone.utc
    )

    off_products = fetch_open_food_facts()

    source_end = datetime.now(
        timezone.utc
    )

    off_count = len(off_products)

    results.append({
        "run_id": RUN_ID,
        "source": "open_food_facts",
        "start_time": source_start.isoformat(),
        "end_time": source_end.isoformat(),
        "record_count": off_count,
        "status": validate_ingestion(
            "open_food_facts",
            off_count
        ),
        "duration_seconds": (
            source_end - source_start
        ).total_seconds()
    })

    inspect_open_food_facts(
        off_products
    )

    # --------------------------------------------------------
    # OPEN PRICES
    # --------------------------------------------------------

    source_start = datetime.now(
        timezone.utc
    )

    open_prices = fetch_open_prices()

    source_end = datetime.now(
        timezone.utc
    )

    open_prices_count = len(
        open_prices
    )

    results.append({
        "run_id": RUN_ID,
        "source": "open_prices",
        "start_time": source_start.isoformat(),
        "end_time": source_end.isoformat(),
        "record_count": open_prices_count,
        "status": validate_ingestion(
            "open_prices",
            open_prices_count
        ),
        "duration_seconds": (
            source_end - source_start
        ).total_seconds()
    })

    inspect_open_prices(
        open_prices
    )

    # --------------------------------------------------------
    # BIGBASKET
    # --------------------------------------------------------

    source_start = datetime.now(
        timezone.utc
    )

    bigbasket_df = load_bigbasket(
        BIGBASKET_FILE
    )

    source_end = datetime.now(
        timezone.utc
    )

    bigbasket_count = (
        len(bigbasket_df)
        if bigbasket_df is not None
        else 0
    )

    results.append({
        "run_id": RUN_ID,
        "source": "bigbasket",
        "start_time": source_start.isoformat(),
        "end_time": source_end.isoformat(),
        "record_count": bigbasket_count,
        "status": validate_ingestion(
            "bigbasket",
            bigbasket_count
        ),
        "duration_seconds": (
            source_end - source_start
        ).total_seconds()
    })

    # --------------------------------------------------------
    # MANIFEST
    # --------------------------------------------------------

    create_ingestion_manifest(
        results
    )

    # --------------------------------------------------------
    # FINAL SUMMARY
    # --------------------------------------------------------

    pipeline_end = datetime.now(
        timezone.utc
    )

    print(
        "\n" + "=" * 60
    )

    print(
        "INGESTION SUMMARY"
    )

    print(
        "=" * 60
    )

    print(
        f"Run ID: {RUN_ID}"
    )

    print(
        f"Total duration: "
        f"{(pipeline_end - pipeline_start).total_seconds():.2f} seconds"
    )

    print(
        f"\nLanding directory:"
    )

    print(
        LANDING_DIR
    )

    print(
        f"\nManifest:"
    )

    print(
        MANIFEST_FILE
    )

    print(
        "\nPipeline completed."
    )


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()

In [0]:
base_path = "/Volumes/fooddata/rawdata/data"

dbutils.fs.mkdirs(f"{base_path}/source")
dbutils.fs.mkdirs(f"{base_path}/landing/bigbasket")
dbutils.fs.mkdirs(f"{base_path}/landing/open_food_facts")
dbutils.fs.mkdirs(f"{base_path}/landing/open_prices")

print("Folder structure created!")

In [0]:
feat: update ingestion for Databricks Volume